# Mapping copy-number events onto branches

Beyond the tree and the genotypes, ScisTreeCNA can report **where** in the lineage each
site gained or lost copies. `map_copy_gain_and_loss` decodes the per-node copy-number
states for a chosen set of loci and annotates every branch with the events on it.

In [1]:
import numpy as np
import scistreecna as scna

reads, cell_names, site_names = scna.util.read_csv('data/test_data_reads.csv')
print(reads.shape, len(cell_names), len(site_names))

(100, 60, 3) 60 100


Event mapping runs on a tree you already have, so first infer one. It could equally be a
tree from another method, or a hypothesis you want to test.

In [2]:
tree, geno = scna.infer(
    reads,
    cell_names=cell_names,
    cn_min=1, cn_max=5,
    tree_batch_size=128,
    verbose=True,
    verbose_mode="min",
)

───────────────────────────────────────── ScisTreeCNA ──────────────────────────────────────────


                                      #Cell: 60 #Site: 100


                   CN_MIN: 1 CN_MAX: 5 ADO: 0.1 SEQ_ERR: 0.01 CN_NOISE: 0.05


                     MAX_ITER: inf TREE_BATCH_SIZE: 128 NODE_BATCH_SIZE: 64


───────────────────────────────────────── Local Search ─────────────────────────────────────────


...

⠙ NNI Searching [Iteration 32]  Likelihood: -12405.9097


⠸ NNI Searching [Iteration 33]  Likelihood: -12405.8828


[15:27:35] Local search converge. Best Likelihood: -12405.882835388184
⠸ NNI Searching [Iteration 33]  Likelihood: -12405.8828


⠸ NNI Searching [Iteration 33]  Likelihood: -12405.8828


────────────────────────────────────────────────────────────────────────────────────────────────


## A label convention you have to know about

Internally ScisTreeCNA identifies each leaf by its **integer cell index as a string**
(`"0"`, `"1"`, ...), and uses that index to look up the cell's row in the likelihood
array. `infer` relabels the tree to your real `cell_names` on the way out, as a
convenience.

`map_copy_gain_and_loss` works at the internal level, so it needs those numeric labels
back. There are two separate requirements, and it is easy to satisfy only the first:

1. Each leaf's **name** must be the integer cell index. Otherwise the lookup
   `int(node.name)` raises `ValueError: invalid literal for int() with base 10: 'c40'`.
2. Each node's **identifier** must equal its name. `relabel` rewrites names but leaves
   identifiers untouched, so a relabelled tree on its own still raises `KeyError: '6'`.

Round-tripping through Newick satisfies both at once, because parsing a Newick string
rebuilds the identifiers from the names.

:::{warning}
This is a sharp edge in the current API rather than something you would guess. If you hit
either of those two errors, this is the reason.
:::

In [3]:
to_index = {name: str(i) for i, name in enumerate(cell_names)}
to_name  = {str(i): name for i, name in enumerate(cell_names)}

# 1. rename leaves to their cell indices; 2. round-trip so identifiers match the names
tree_numeric = scna.util.from_newick(str(scna.util.relabel(tree, name_map=to_index)))

print("infer() returned leaf names:", sorted(tree.get_leaves(return_label=True))[:5])
print("after relabel + round-trip :",
      sorted(tree_numeric.get_leaves(return_label=True), key=int)[:5])
print("identifier == name?        :",
      all(tree_numeric[l].name == l for l in tree_numeric.get_leaves()))

infer() returned leaf names: ['c0', 'c1', 'c10', 'c11', 'c12']
after relabel + round-trip : ['0', '1', '2', '3', '4']
identifier == name?        : True


## Mapping events

`map_copy_gain_and_loss` takes the reads, the tree, and the loci to map. Loci are given by
**name**, so `site_names` has to be passed as well.

Two parameters control what counts as an event:

- `allele` — which allele to track: `0` for wild-type copies, `1` (the default) for mutant
  copies.
- `loh` — when `True` (the default), only losses reaching **zero** copies are reported,
  i.e. true loss of heterozygosity. Set `False` to report every decrease.

In [4]:
loci = site_names[:20]

mapped = scna.map_copy_gain_and_loss(
    reads,
    tree_numeric,
    loci=loci,
    cell_names=cell_names,
    site_names=site_names,
    cn_min=1, cn_max=5,
    ado=0.1, seq_error=0.01, cn_noise=0.05,
    allele=1,     # track the mutant allele
    loh=True,     # only losses down to zero
)
type(mapped)

scistreecna.base.tree.BaseTree

## Reading the result

The return value is a tree whose nodes each carry an `events` dict listing the loci that
gained or lost copies on the branch leading to that node.

In [5]:
traversor = scna.util.TraversalGenerator()

n_gain = n_loss = 0
rows = []
for node in traversor(mapped):
    gains, losses = node.events['gain'], node.events['loss']
    n_gain += len(gains)
    n_loss += len(losses)
    if gains or losses:
        label = f"cell {to_name[node.name]}" if node.is_leaf() else f"internal {node.name}"
        rows.append((label, gains, losses))

print(f"branches carrying events: {len(rows)}")
print(f"total gain events: {n_gain}, total loss events: {n_loss}\n")

for label, gains, losses in rows[:15]:
    print(f"{label:>16}   gain: {gains}   loss: {losses}")

branches carrying events: 14
total gain events: 18, total loss events: 0

        cell c40   gain: ['s8']   loss: []
internal 3f1b79987f   gain: ['s12']   loss: []
internal f81db9b013   gain: ['s9']   loss: []
        cell c32   gain: ['s3', 's18']   loss: []
internal ce5c4f7346   gain: ['s19']   loss: []
internal 72d8a1460a   gain: ['s4', 's13']   loss: []
        cell c14   gain: ['s16']   loss: []
         cell c6   gain: ['s1', 's15']   loss: []
internal cbf4a9cd55   gain: ['s11', 's14']   loss: []
        cell c17   gain: ['s2']   loss: []
        cell c19   gain: ['s7']   loss: []
        cell c16   gain: ['s17']   loss: []
internal d2e4250bbf   gain: ['s10']   loss: []
internal ff8620d350   gain: ['s6']   loss: []


Where an event sits matters as much as that it happened. Events on branches near the root
are shared by every cell below them — early, clonal changes. Events on terminal branches
are private to a single cell. That contrast is the point of the exercise.

In [6]:
leaf_branches = sum(1 for n in traversor(mapped)
                    if n.is_leaf() and (n.events['gain'] or n.events['loss']))
internal_branches = sum(1 for n in traversor(mapped)
                        if not n.is_leaf() and (n.events['gain'] or n.events['loss']))

print(f"terminal (cell-private) branches with events: {leaf_branches}")
print(f"internal (shared)        branches with events: {internal_branches}")

terminal (cell-private) branches with events: 7
internal (shared)        branches with events: 7


## How strict should a "loss" be?

`loh=True` only counts a decrease that goes all the way to **zero** copies — a true loss
of heterozygosity. `loh=False` counts every decrease, a much looser definition that
reports many more events.

In [7]:
mapped_any = scna.map_copy_gain_and_loss(
    reads, tree_numeric,
    loci=loci,
    cell_names=cell_names, site_names=site_names,
    cn_min=1, cn_max=5,
    allele=1,
    loh=False,     # every decrease counts, not just losses down to zero
)

any_loss = sum(len(n.events['loss']) for n in traversor(mapped_any))
print(f"loss events with loh=True  (down to zero only): {n_loss}")
print(f"loss events with loh=False (any decrease)     : {any_loss}")

loss events with loh=True  (down to zero only): 0
loss events with loh=False (any decrease)     : 0


## Tracking the wild-type allele instead

`allele=0` tracks copies of the **wild-type** allele, so a "loss" is deletion of the
reference copy — the event behind loss of heterozygosity in the usual sense.

In [8]:
mapped_wt = scna.map_copy_gain_and_loss(
    reads, tree_numeric,
    loci=loci,
    cell_names=cell_names, site_names=site_names,
    cn_min=1, cn_max=5,
    allele=0,     # wild-type allele
    loh=True,
)

wt_gain = sum(len(n.events['gain']) for n in traversor(mapped_wt))
wt_loss = sum(len(n.events['loss']) for n in traversor(mapped_wt))
print(f"wild-type allele — gain events: {wt_gain}, loss events: {wt_loss}")

wild-type allele — gain events: 36, loss events: 1


:::{tip}
Mapping every site at once is rarely what you want, and it is not cheap. Pick the loci
that matter — sites in known driver genes, or ones whose calls disagree between methods —
and map those.
:::

## Related

- [](allele_specific.ipynb) — allele-specific copy numbers, which make copy-number-neutral
  LOH identifiable in the first place
- [](evaluation.ipynb) — checking inferred trees against a known truth